# Divergence figures

Visualization of the divergence comparison, from the saved results. `INDIR` below picks where
they are read from.

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.decomposition import PCA


def _bootstrap():
    here = os.path.abspath(os.path.dirname(__file__)) if "__file__" in globals() else os.getcwd()
    # this folder (its helper modules) + tools/ (the shared `_repo.py` path resolver)
    for d in (here, os.path.abspath(os.path.join(here, os.pardir, os.pardir, "tools"))):
        if d not in sys.path:
            sys.path.insert(0, d)
    import _repo
    _repo.add_paths()
    return _repo


P = _bootstrap()
# the shipped paper artifacts; switch to P.results("divergence") to prefer your own
# `new_results/` re-run of the two experiment notebooks
INDIR = P.shipped("divergence")

# ---- TEXT SIZE -------------------------------------------------------------------------------
# Same scheme as the outlier-simulation figures. The figures go into the draft at
# `width=\linewidth` (= 6.5in), so LaTeX rescales them by 6.5 / figsize-width and what the reader
# sees is  printed pt = fontsize x 6.5 / figsize-width  --  0.36x for the outlier row (18in wide),
# 0.54x for the real-data 2x3 (12in). FS multiplies every font size at once; the per-element sizes
# stay written out next to the thing they control, so any one of them can be nudged on its own.
FS = globals().get("FS", 1.6)
plt.rcParams.update({"font.size": 11 * FS, "axes.titlesize": 11 * FS, "axes.labelsize": 11 * FS,
                     "figure.titlesize": 13 * FS, "xtick.labelsize": 10 * FS,
                     "ytick.labelsize": 10 * FS, "legend.fontsize": 10 * FS})

BARY_C  = "#7b3fa0"     # barycenter / predicted cloud (outlier sim)
PRED_C  = "tab:purple"  # predicted cloud (real data)
TRUE_C  = "0.72"        # held-out ground truth (real data)
STAR_C, X_C = "#08306b", "#67000d"      # inlier / outlier mode markers
N_BG, N_BARY = 1500, 700                # points DRAWN: grey background, barycenter cloud

## Outlier simulation: one panel per divergence

In [ ]:
def fig_outliers(samples, metrics, pooled, inlier_means=None, outlier_means=None, tau=None,
                 name="outliers_fig"):
    """`inlier_means` / `outlier_means` are accepted but no longer drawn (the mode markers were
    dropped); they are kept in the signature so existing call sites do not break."""
    Pp = PCA(2).fit(pooled).transform
    rng = np.random.default_rng(0)
    BG = Pp(pooled[rng.choice(len(pooled), min(N_BG, len(pooled)), replace=False)])
    labels = list(samples.keys())

    # height 4.0 -> 4.4: titles/labels are fixed in points, so at FS=1.6 they would otherwise eat
    # the panels (the width is what sets the printed text size, so it stays put).
    fig, axes = plt.subplots(1, len(labels), figsize=(3.6 * len(labels), 4.4), dpi=140,
                             sharex=True, sharey=True)
    for ax, label in zip(np.atleast_1d(axes), labels):
        Xb = np.asarray(samples[label])
        Xb = Xb[rng.choice(len(Xb), min(N_BARY, len(Xb)), replace=False)]
        Xp = Pp(Xb)
        ax.scatter(BG[:, 0], BG[:, 1], s=5, c="0.85", alpha=0.4)
        ax.scatter(Xp[:, 0], Xp[:, 1], s=6, c=BARY_C, alpha=0.55)
        ax.set_title(label, fontsize=11 * FS)
        ax.set_xlabel("PC1")
        # one-entry legend carrying the metric; 9*FS + tight padding so the box stays well inside a
        # panel that is only ~1/5 of the row
        ax.legend(handles=[Line2D([0], [0], marker="o", ls="", mfc=BARY_C, mec="none",
                                  label=f"W2 to ideal = {metrics[label]['w2']:.2f}")],
                  fontsize=9 * FS, loc="upper right", handlelength=1.0, handletextpad=0.4,
                  borderpad=0.35, borderaxespad=0.4)
    np.atleast_1d(axes)[0].set_ylabel("PC2")
    # headroom so the legend sits on white space rather than on the cloud (shared axes -> set once)
    y0, y1 = np.atleast_1d(axes)[0].get_ylim()
    np.atleast_1d(axes)[0].set_ylim(y0, y1 + 0.22 * (y1 - y0))

    fig.tight_layout(rect=[0, 0, 1, 0.93])
    fig.suptitle(f"Outlier simulation: barycenter by divergence (tau={tau})",
                 fontsize=13 * FS, y=0.985)
    return fig

## Real data: true held-out cells vs each divergence's prediction

In [ ]:
def fig_realdata(truth, preds, res, dataset, dim, held, name=None, n_pred=None):
    methods = [m for m in preds if m not in ("naive-midpoint", "carry-forward", "truth")]
    rng = np.random.default_rng(0)
    n_pred = n_pred or max(len(np.asarray(preds[m])) for m in methods)
    truth = np.asarray(truth)
    true_sub = truth[rng.integers(0, len(truth), min(int(1.2 * n_pred), len(truth)))]
    panels = [("true held-out", None)] + [(m, np.asarray(preds[m])) for m in methods]

    # height 7.6 -> 8.2 for the same reason as above (fixed-point titles vs bigger type)
    fig, axes = plt.subplots(2, 3, figsize=(12, 8.2), dpi=140, sharex=True, sharey=True)
    for i, ax in enumerate(axes.ravel()):
        if i >= len(panels):
            ax.axis("off"); continue
        nm, g = panels[i]
        ax.scatter(true_sub[:, 0], true_sub[:, 1], s=6, c=TRUE_C, alpha=0.45)
        if g is None:
            ax.set_title("true held-out", fontsize=11 * FS)
        else:
            ax.scatter(g[:, 0], g[:, 1], s=6, c=PRED_C, alpha=0.55)
            ax.set_title(f"{nm}  (W2 {_w2_of(res, nm):.2f})", fontsize=11 * FS)
    for ax in axes[:, 0]:
        ax.set_ylabel("PC2")
    for ax in axes[-1, :]:
        ax.set_xlabel("PC1")

    # header stack: suptitle -> legend row -> panel titles -> panels
    fig.tight_layout(rect=[0, 0, 1, 0.90])
    fig.suptitle(f"{dataset} d={dim}, held Day {held}: predicted vs true held-out cells",
                 fontsize=13 * FS, y=0.985)
    fig.legend(handles=[
        Line2D([0], [0], marker="o", ls="", mfc=PRED_C, mec="none", label="estimated"),
        Line2D([0], [0], marker="o", ls="", mfc=TRUE_C, mec="none", label="ground truth (held-out)")],
        loc="upper center", ncol=2, frameon=False, fontsize=10 * FS, bbox_to_anchor=(0.5, 0.945))
    return fig


def _w2_of(res, method):
    """`res` is either the in-notebook dict {method: {'W2': [per-seed, ...]}} or the saved
    metrics json {method: {'W2': {'mean':..,'std':..}}}."""
    v = res[method]["W2"]
    return float(v["mean"]) if isinstance(v, dict) else float(np.mean(v))

## The figures

In [ ]:
if __name__ == "__main__":
    with open(os.path.join(INDIR, "outliers_metrics.json")) as f:
        om = json.load(f)
    z = np.load(os.path.join(INDIR, "outliers_samples.npz"))
    samples = {k.replace("_", " "): z[k] for k in z.files
               if k not in ("pooled", "target", "inlier_means", "outlier_means")}
    samples = {k: samples[k] for k in om if k in samples}          # keep the metrics-file order
    fig_outliers(samples, om, z["pooled"], z["inlier_means"], z["outlier_means"],
                 tau= 1.0)

    for ds, dim, held in [("embryoid", 20, "13.5"), ("statefate", 20, "4.0")]:
        tag = f"realdata_{ds}{dim}_Day{held}"
        npz = np.load(os.path.join(INDIR, f"{tag}_preds.npz"), allow_pickle=True)
        preds = {k.replace("_", " "): npz[k] for k in npz.files}    # npz uses "UOT_kl", metrics "UOT kl"
        with open(os.path.join(INDIR, f"{tag}_metrics.json")) as f:
            res = json.load(f)
        truth = preds.pop("truth")
        preds = {k: preds[k] for k in res if k in preds}            # metrics-file order
        fig_realdata(truth, preds, res, ds, dim, held, name=f"{tag}_fig")